In [1]:
import pathlib
FDIR_APP = pathlib.Path(".").absolute()
from constants import PTH_EUI, PTH_BUILDING_TYPES
PTH_EUI_DATA = FDIR_APP / PTH_EUI
PTH_BUILDING_TYPES = FDIR_APP / PTH_BUILDING_TYPES
from aectemplater_ui import ENV  # TODO: This changes the cwd. is that necessary here?
import sys
sys.path.append("/home/jovyan/poe")

In [4]:
# BUILDING_TYPES = PTH_BUILDING_TYPES.read_text().split("\n")

In [2]:
# ENV.model_dump()

In [5]:
import pandas as pd
import numpy as np
import ipywidgets as w
import traitlets as tr
from markdown import markdown
from IPython.display import IFrame
import warnings
import duckdb
import altair as alt
import stringcase

from ipydatagrid import VegaExpr
from ipyautoui.custom.buttonbars import CrudOptions, CrudView
from ipyautoui.autodisplay_renderers import preview_vegalite_json
from ipyautoui.custom import FileDownload

from aectemplater_client import get_object_by_code, get_type_spec_object_data_grid, post_type_spec_data, get_instance_specs_object_data_grid, get_object_gridschema, get_project_by_project_number
from aectemplater_ui.type_specification import TypeSpecGrid
from aectemplater_ui.instance_specification import InstanceSpecGrid, InstanceSpecDelete
from aectemplater_ui.object_specification import ObjectSpecGrid
from aectemplater_ui.load_project import LoadProject, AbstractAppTemplate
from aectemplater_ui import ENV  # TODO: This changes the cwd. is that necessary here?
from aectemplater_ui.utils import stretch_tab_widths


from ui import RecordedEnergyUse

def get_object_id():
    return get_object_by_code(code="MxfProjectBuildingArea")["id"]

OBJECT_ID_BUILDING_AREA = get_object_id()
ANNUAL_ENERGY_DASHBOARD_URL = "https://wiki.maxfordham.com/poe/index"
EUI_DATA = pd.read_csv(PTH_EUI_DATA)
stretch_tab_widths()
warnings.filterwarnings("ignore")

In [6]:
# value = FDIR_APP / "2-835.xlsx"

# f = FileDownload(value=value)
# f.reload()
# f

In [7]:
# import ipydatagrid as dg

# _ = {"Abbreviation": 100, "TypeReference": 100, "BenchmarkTargetYear":150, "Id": 5}
# map_ = {'Abbreviation': ('Identity Data', 'Abbreviation', ''),
#  'TypeReference': ('Identity Data', 'Type Reference', ''),
#  'BenchmarkTargetYear': ('Identity Data', 'Benchmark Target Year', ''),
#  'Id': ('Undefined', 'Id', '')}
# column_widths = {map_[k]: v for k, v in _.items()}
# data = {map_[k]: [v] for k, v in _.items()}
# # bldgs.grid.column_widths = {bldgs.grid.map_name_index[k]: v for k,v in _.items()}

# gr = dg.DataGrid(pd.DataFrame(data))
# gr.column_widths = column_widths
# gr

In [8]:
class BldgsGrid(TypeSpecGrid):
    def __init__(self, **kwargs):
        kwargs = kwargs | {"object_id": OBJECT_ID_BUILDING_AREA}
        super().__init__(**kwargs)
        # self.transposed = False
        # _ = {"Abbreviation": 100, "TypeReference": 100, "BenchmarkTargetYear":100, "Id": 1}
        # self.grid.column_widths = {self.grid.map_name_index[k]: v for k,v in _.items()}
        self.buttonbar_grid.imports.layout.display = "none"
        self.buttonbar_grid.images.layout.display = "none"
        
class AreasGrid(InstanceSpecGrid):
    def __init__(self, **kwargs):
        kwargs = kwargs | {"object_id": OBJECT_ID_BUILDING_AREA}
        super().__init__(**kwargs)

In [9]:
def get_eui_targets():
    metric = "kWh/m²GIA/yr"
    li = [" ".join(x.split("-")) for x in EUI_DATA.columns]
    li = [stringcase.pascalcase(stringcase.snakecase(x)) for x in li]
    df_eui = EUI_DATA[EUI_DATA.unit == metric].copy(deep=True)
    df_eui.columns = li
    return df_eui

def get_area_data(project_revision_id, OBJECT_ID_BUILDING_AREA):
    _ = get_instance_specs_object_data_grid(project_revision_id, OBJECT_ID_BUILDING_AREA)
    data, schema = _["data"], _["$schema"]
    map_new_build = {True: "newbuild", False: "retrofit-in-one-go"}
    df_area_data = pd.DataFrame(data)
    df_area_data["BuildingType"] = df_area_data.BuildingType
    df_area_data['ConstructionDeliveryType'] = df_area_data.IsNewBuild.map(map_new_build)
    return df_area_data

def get_area_summary(df_area_data):
    return pd.pivot_table(df_area_data, index=["BuildingType", "ConstructionDeliveryType"], columns="TypeMark", values="GrossInternalArea", aggfunc="sum")

def get_area_weights_dict(df_area_weight):
    di_area_wt = {}
    for x in df_area_weight.columns:
        di_area_wt[x] = np.round(df_area_weight[x] / df_area_weight[x].sum(), 2).dropna().to_dict()
    return di_area_wt

def get_blnd_targets(df_eui, di_area_wt):
    q = """
    SELECT * 
    FROM df_eui
    WHERE
    BuildingType = '{b_type}'
    AND
    ConstructionDeliveryType = '{c_type}'
    """
    
    df_blnd_bmarks = pd.DataFrame()
    for k, v in di_area_wt.items():
        df_tmp = sum([duckdb.sql(q.format(b_type=k_[0], c_type=k_[1])).fetchdf().set_index("Year")[["BenchmarkTarget"]] * v_ for k_, v_ in v.items()])
        df_tmp["BuildingTypeShorthand"] = k
        df_tmp["BuildingType"] = str(v)
        df_tmp = df_tmp.reset_index()
        df_blnd_bmarks = pd.concat([df_blnd_bmarks, df_tmp])
    
    return df_blnd_bmarks

def update_area_weighted_targets(project_revision_id, df_eui, object_id=OBJECT_ID_BUILDING_AREA):
    df_area_data = get_area_data(project_revision_id, OBJECT_ID_BUILDING_AREA)
    df_area_summary = get_area_summary(df_area_data)
    di_area_weight = get_area_weights_dict(df_area_summary)
    df_blnd_bmarks = get_blnd_targets(df_eui, di_area_weight)
    return df_area_data, df_area_summary, di_area_weight, df_blnd_bmarks

In [10]:
def get_vis(df_area_summary, df_blnd_bmarks):
    df_area_summary_styled = df_area_summary.fillna(0).style.bar(
        subset=list(df_area_summary.columns), 
        color='lightblue').format('{:.0f} m2')
    ch = alt.Chart(df_blnd_bmarks).mark_circle(size=60).encode(
        x='BenchmarkTarget',
        y='BuildingTypeShorthand',
        color=alt.Color("Year").scale(scheme="redblue"),
    )
    out_area = w.Output()
    out_bmark = w.Output()
    with out_area: 
        display(df_area_summary_styled)
    with out_bmark: 
        display(ch)
    return w.HBox([out_area, out_bmark])

def update_summary(tabs, on_change=None):
    h1 = "# Net-Zero Carbon Project Summary"
    h2 = "## Summary of Project Building Areas"
    h3 = "## Project Area-Weighted Energy Use Intensity Targets"
    bn_print = w.Button(icon="print", layout={"width":"40px"})
    bn_download_xl = FileDownload(value=(FDIR_APP / "2-835.xlsx"))
    bn_download_xl.reload()
    vbx_summary = w.VBox(
        [
            w.HBox([
                w.HTML(markdown(h1)),
                bn_print,
                bn_download_xl
            ]),
            w.VBox([
                w.HTML(markdown(h2)),
                out_areas
            ]),
            w.VBox([
                w.HTML(markdown(h3)),
                out_bmarks
            ])
        ]
    )
    
    tabs.children[tab_titles.index("Summary")].children = [vbx_summary]

In [11]:
project_revision_id = get_project_by_project_number(ENV.AECSCHEDULE_PROJECT_NUMBER)["id"]

# title 
html_title = w.HTML(markdown("# Net-Zero Carbon Metrics"))

# project loader
select_project = LoadProject(fn_onclick=lambda: print("load project"))  # self.fn_onclick
hbx_title = w.HBox([html_title, select_project], layout=w.Layout(justify_content="space-between"))

# building
bldgs = BldgsGrid(project_revision_id=project_revision_id)

# building areas
areas = AreasGrid(object_id=OBJECT_ID_BUILDING_AREA, project_revision_id=project_revision_id)

# energy use
reu = RecordedEnergyUse()

# benchmark data
# out_eui_plot = w.Output()
# with out_eui_plot:
#     preview_vegalite_json("eui-targets.json")
img_eui_plot = w.Image.from_file(FDIR_APP / "eui-targets.png", layout=dict(width="1000px"))


tab_titles = ["Summary", "Project Buildings", "Building Areas", "Predicted Operational Energy", "Recorded In-Use Operational Energy", "Embodied Carbon", "REFERENCE - Energy Use Intensity Targets"]
tabs = w.Tab([w.VBox([w.HTML(f"{x}".format(x))]) for x in tab_titles], titles=tab_titles)


tabs.children[tab_titles.index("Project Buildings")].children = [bldgs]
tabs.children[tab_titles.index("Building Areas")].children = [areas]
tabs.children[tab_titles.index("Recorded In-Use Operational Energy")].children = [reu]
tabs.children[tab_titles.index("REFERENCE - Energy Use Intensity Targets")].children = [img_eui_plot]

df_eui = get_eui_targets()
df_area_data, df_area_summary, di_area_weight, df_blnd_bmarks = update_area_weighted_targets(project_revision_id, df_eui, object_id=OBJECT_ID_BUILDING_AREA)

df_area_summary_styled = df_area_summary.fillna(0).style.bar(
    subset=list(df_area_summary.columns), 
    color='lightblue').format('{:.0f} m2')
ch = alt.Chart(df_blnd_bmarks).mark_circle(size=60).encode(
    x='BenchmarkTarget',
    y='BuildingTypeShorthand',
    color=alt.Color("Year").scale(scheme="redblue"),
)
out_areas = w.Output()
out_bmarks = w.Output()
with out_areas: 
    display(df_area_summary_styled)
with out_bmarks: 
    display(ch)

update_summary(tabs)
tabs.observe(update_summary, "selected_index")

app = w.VBox([hbx_title, tabs])
app

reload


In [10]:
# HACK: why can't I do this in the __init__
_ = {"Abbreviation": 100, "TypeReference": 100, "BenchmarkTargetYear":100, "Id": 2}
bldgs.grid.column_widths = {bldgs.grid.map_name_index[k]: v for k,v in _.items()}